# Išlaidų reikalavimo analizė

Šiame užraše demonstruojama, kaip sukurti agentus, naudojančius papildinius kelionės išlaidoms iš vietinių kvitų vaizdų apdoroti, sugeneruoti išlaidų reikalavimo el. laišką ir vizualizuoti išlaidų duomenis naudojant pyrago diagramą. Agentai dinamiškai pasirenka funkcijas pagal užduoties kontekstą.

Žingsniai:
1. OCR agentas apdoroja vietinį kvito vaizdą ir ištrina kelionės išlaidų duomenis.
2. El. pašto agentas sugeneruoja išlaidų reikalavimo el. laišką.

### Kelionės išlaidų scenarijaus pavyzdys:
Įsivaizduokite, kad esate darbuotojas, keliaujantis į verslo susitikimą kitame mieste. Jūsų įmonės politika yra kompensuoti visas pagrįstas su kelione susijusias išlaidas. Štai galimų kelionės išlaidų suskirstymas:
- Transportas:
Skrydžio bilietai į abi puses iš jūsų gyvenamojo miesto į tikslinį miestą.
Taksi ar automobilio nuomos paslaugos iki ir nuo oro uosto.
Vietinis transportas tikslo mieste (pavyzdžiui, viešasis transportas, automobilių nuoma ar taksi).

- Apgyvendinimas:
Viešbučio viešnagė tris naktis vidutinės klasės verslo viešbutyje netoli susitikimo vietos.

- Maitinimas:
Dienos maitinimo išlaidos pusryčiams, pietums ir vakarienei, remiantis įmonės dienpinigiais.

- Kitos išlaidos:
Automobilio stovėjimo mokestis oro uoste.
Internetą suteikiančios paslaugos mokestis viešbutyje.
Arbatpinigiai arba nedideli paslaugų mokesčiai.

- Dokumentacija:
Jūs pateikiate visus kvitus (skrydžiai, taksi, viešbutis, maitinimas ir kt.) ir užpildytą išlaidų ataskaitą kompensacijai.


## Reikalingų bibliotekų importavimas

Importuokite būtinas bibliotekas ir modulius šiam užrašų knygeliui.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Apibrėžkite Išlaidų Modelius

 Sukurkite Pydantic modelį atskiroms išlaidoms ir ExpenseFormatter klasę, kuri paverstų vartotojo užklausą į struktūruotus išlaidų duomenis.

 Kiekviena išlaida bus pateikta formatu:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Įrankių apibrėžimas – el. laiško generavimas

Sukurkite įrankio funkciją, skirtą sugeneruoti el. laišką išlaidų deklaracijai pateikti.
- Šis įrankis naudoja `@tool` dekoratorių iš Microsoft Agent Framework.
- Jis apskaičiuoja bendrą išlaidų sumą ir suformuoja detales į el. laiško turinį.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Įrankis kelionės išlaidoms iš kvito vaizdų išgauti

Sukurkite įrankio funkciją, skirtą kelionės išlaidoms iš kvito vaizdų išgauti.  
- Šis įrankis naudoja `@tool` dekoratorių iš Microsoft Agent Framework.  
- Jis nuskaito kvito vaizdą, užkoduoja jį kaip base64 ir grąžina duomenų URI agentui analizei.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Išlaidų apdorojimas

Apibrėžkite agentus ir sujunkite juos į seką naudodami `WorkflowBuilder`.
- OCR agentas iš iškvito vaizdo naudoja `load_receipt_image` įrankį, kad išgautų struktūrizuotus išlaidų duomenis.
- El. pašto agentas paima išgautus duomenis ir sukuria profesionalų išlaidų kompensavimo el. laišką naudodamas `generate_expense_email` įrankį.
- `WorkflowBuilder` su `add_edge` sukuria seką: OCR agentas → El. pašto agentas.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Pagrindinė funkcija

Sukurkite sekimų srautą ir paleiskite jį, kad apdorotumėte kvito vaizdą ir sugeneruotumėte išlaidų kompensacijos el. laišką.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatizuoti vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas gimtąja kalba turėtų būti laikomas autoritetingu šaltiniu. Kritinei informacijai rekomenduojama naudoti profesionalius žmogaus atliekamus vertimus. Mes neatsakome už bet kokius nesusipratimus ar neteisingus aiškinimus, atsiradusius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
